In [1]:
import sys
sys.path.append("../benchmarks/")

from guacamol.common_scoring_functions import (
    SyntheticAccessibilityScoringFunction,
    CNS_MPO_ScoringFunction
)

import pandas as pd

# Load ../data/train.csv
train = pd.read_csv("../data/train.smi", header=None)
train.columns = ["SMILES"]
print(train.shape)

train.sample(5)

(1584663, 1)


,SMILES
568215,CCN(CC)C(=O)CNc1ccccc1COC
574840,COC(=O)C(C)CN(C)C(=O)c1ccc(F)cc1Cl
987747,COc1ccccc1CC(C)(C)NC(=O)c1ccoc1
1384877,CNC(=O)C(NC(=O)c1c(C)cc(C)cc1C)c1ccc(F)c(F)c1
440612,CCOc1ccc(OCC)c(NC(=O)c2cc(C3CC3)on2)c1


In [2]:
from smiles_utils import filter_and_canonicalize, AllowedSmilesCharDictionary, split_charged_mol

def clean_smile_list(smiles_list):
    raw_smiles = []
    smiles_char_dict = AllowedSmilesCharDictionary()

    for smiles in smiles_list:
        # only keep reasonably sized molecules
        if 5 <= len(smiles) <= 200:

            smiles = split_charged_mol(smiles)

            if smiles_char_dict.allowed(smiles):
                # check whether the molecular graph consists of
                # multiple connected components (eg. in salts)
                # if so, just keep the largest one
                raw_smiles.append(smiles)
    
    filtered_smiles = [filter_and_canonicalize(smiles) for smiles in raw_smiles]
    all_good_mols = sorted(list(set([item[0] for item in filtered_smiles if item])))

    return all_good_mols


smiles_char_dict = AllowedSmilesCharDictionary()

def clean_smile(smiles):
    # only keep reasonably sized molecules
    if 5 > len(smiles) or len(smiles) > 200:
        return None
    
    smiles = split_charged_mol(smiles)

    if not smiles_char_dict.allowed(smiles):
        return None
    
    filtered_smiles = filter_and_canonicalize(smiles)
    return filtered_smiles[0] if filtered_smiles else None

In [3]:
from pandarallel import pandarallel

pandarallel.initialize(progress_bar=True)
train["Clean-SMILES"] = train["SMILES"].parallel_apply(clean_smile)

train.sample(5)

INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


,SMILES,Clean-SMILES
64694,CCCCC(=O)Nc1cccc(-c2nn3c(C)nnc3s2)c1,CCCCC(=O)Nc1cccc(-c2nn3c(C)nnc3s2)c1
635108,Cc1noc(C)c1CN(C)C(=O)c1ccc2c(c1)NC(=O)CO2,Cc1noc(C)c1CN(C)C(=O)c1ccc2c(c1)NC(=O)CO2
866110,COCc1nc(C(=O)Oc2ccc3c(c2)CCC3)cs1,COCc1nc(C(=O)Oc2ccc3c(c2)CCC3)cs1
1022674,CCN(CCC#N)CC(=O)N(C)CC12CC3CC(CC(C3)C1)C2,CCN(CCC#N)CC(=O)N(C)CC12CC3CC(CC(C3)C1)C2
1226199,O=C1C(Sc2nc3ncccn3n2)CCCN1c1cccc(F)c1,O=C1C(Sc2nc3ncccn3n2)CCCN1c1cccc(F)c1


In [4]:
import sys
import os
from rdkit import Chem

sys.path.append(os.path.join(Chem.RDConfig.RDContribDir, 'SA_Score'))
import sascorer

sa_scorer = SyntheticAccessibilityScoringFunction(sascorer.calculateScore)
cns_mpo_scorer = CNS_MPO_ScoringFunction()

train["SA-Score"] = train["Clean-SMILES"].parallel_apply(sa_scorer.raw_score)
train["CNS-MPO-Score"] = train["Clean-SMILES"].parallel_apply(cns_mpo_scorer.raw_score)

train.sample(5)

,SMILES,Clean-SMILES,SA-Score,CNS-MPO-Score
688584,Cc1nc(C(=O)C(C#N)c2nc3ccccc3o2)cs1,Cc1nc(C(=O)C(C#N)c2nc3ccccc3o2)cs1,0.764865,1.000000
72342,N#Cc1cnc(SCC(=O)c2ccc3c(c2)NC(=O)CO3)nc1N,N#Cc1cnc(SCC(=O)c2ccc3c(c2)NC(=O)CO3)nc1N,0.816879,0.799946
1067506,O=C(Nc1ccc(CO)cn1)c1ccc(F)c(F)c1F,O=C(Nc1ccc(CO)cn1)c1ccc(F)c(F)c1F,0.872774,0.921306
185831,CSc1nc2ncc(C(=O)OC(C)C)c(C)n2n1,CSc1nc2ncc(C(=O)OC(C)C)c(C)n2n1,0.825084,1.000000
1158527,Cn1c(=O)oc2cc(NC(=O)NCC(=O)N3CCCCC3)ccc21,Cn1c(=O)oc2cc(NC(=O)NCC(=O)N3CCCCC3)ccc21,0.862709,0.916553


In [5]:
top_sascore = train.nlargest(5000, "SA-Score")
top_cnsmpo = train.nlargest(5000, "CNS-MPO-Score")

print(top_sascore["SA-Score"].describe())
print(top_cnsmpo["CNS-MPO-Score"].describe())

top_sascore_smiles = top_sascore["Clean-SMILES"].tolist()
top_cnsmpo_smiles = top_cnsmpo["Clean-SMILES"].tolist()

print(top_sascore_smiles[:5])
print(top_cnsmpo_smiles[:5])

count    5000.000000
mean        0.942585
std         0.005408
min         0.936524
25%         0.938382
50%         0.941041
75%         0.945336
max         0.970067
Name: SA-Score, dtype: float64
count    5000.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: CNS-MPO-Score, dtype: float64
['COc1ccc(NC(=O)c2ccc(OC)cc2)cc1', 'COC(=O)c1ccc(C(=O)Nc2ccccc2)cc1', 'COC(=O)c1ccc(NC(=O)c2ccccc2)cc1', 'O=C(COc1ccc(Cl)cc1)Nc1ccccc1', 'O=C(CNC(=O)c1ccccc1)Nc1ccccc1']
['CC(C)(C)C(=O)C(Oc1ccc(Cl)cc1)n1ccnc1', 'CCOC(=O)c1cncn1C1CCCc2ccccc21', 'COc1ccccc1OC(=O)Oc1ccccc1OC', 'CC1CC(OC(=O)CN2CCCC2=O)CC(C)(C)C1', 'O=C(C1CCCCC1)N1CC(=O)N2CCc3ccccc3C2C1']


In [6]:
# Load data/BBB.csv
import pandas as pd
from rdkit import Chem

bbb = pd.read_csv("../data/BBB.csv")

# Create SMILES from InChI
bbb["SMILES"] = bbb["InChI"].apply(lambda x: Chem.MolToSmiles(Chem.inchi.MolFromInchi(x)))
active_bbb = bbb.loc[
    (bbb["activity"] == "active") &
    (bbb["corrupted"] == False),
    ["SMILES", "activity"]
]

print(active_bbb.shape)
active_bbb.sample(5)

(4956, 2)


,SMILES,activity
3349,COC(=O)c1cncn1[C@@H](C)c1ccccc1,active
4004,COc1ccc2c(c1OC)C(=O)O[C@H]2Nc1ccc2c(c1)COC2=O,active
4312,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,active
4298,CC1CC2C3C(Cl)CC4=CC(=O)C=CC4(C)C3C(O)CC2(C)C1(...,active
3914,COC(=O)C[C@H](C1=C(O)C(=O)C([C@@H](CC(=O)OC)c2...,active


In [8]:
from sklearn.model_selection import train_test_split

bbb_for_training = bbb.assign(
    activity=[1 if x == "active" else 0 for x in bbb["activity"]]
)

bbb_for_training, bbb_for_testing = train_test_split(bbb_for_training, test_size=0.2, random_state=42)
bbb_for_training, bbb_for_validation = train_test_split(bbb_for_training, test_size=0.1, random_state=42)
bbb_for_training["split"] = "train"
bbb_for_validation["split"] = "val"
bbb_for_testing["split"] = "test"

bbb_dataset = pd.concat([bbb_for_training, bbb_for_validation, bbb_for_testing])
bbb_dataset[["SMILES", "activity", "split"]].to_csv("../qsar/data/bbb_for_training.csv", index=False)

In [9]:
# Load data/excape_all_active_compounds.smi
all_active_compounds = pd.read_csv(
    "../data/excape_all_active_compounds.smi", header=None, names=["SMILES", "target"], sep="\t"
)
max_target = all_active_compounds["target"].max()

print(max_target)

426


In [10]:
sascore_idx = max_target + 1
cnsmpo_idx = max_target + 2
bbb_idx = max_target + 3

top_sascore_df = pd.DataFrame({
    "SMILES": top_sascore_smiles,
    "target": sascore_idx
})

top_cnsmpo_df = pd.DataFrame({
    "SMILES": top_cnsmpo_smiles,
    "target": cnsmpo_idx
})

top_bbb_df = pd.DataFrame({
    "SMILES": bbb["SMILES"].tolist(),
    "target": bbb_idx
})

top_df = pd.concat([
    all_active_compounds, 
    top_sascore_df, 
    top_cnsmpo_df, 
    top_bbb_df
]).sample(frac=1, random_state=42)

top_df.to_csv("../data/active_compounds_mpo.smi", sep="\t", index=False, header=False)

In [13]:
import json

with open("../data/target_conditions_to_index.json", "r") as f:
    target_conditions_to_index = json.load(f)

idx_to_gene = target_conditions_to_index["idx_to_gene"]
gene_to_index = target_conditions_to_index["gene_to_index"]

idx_to_gene[int(sascore_idx)] = "SAScore"
idx_to_gene[int(cnsmpo_idx)] = "CNSMPO"
idx_to_gene[int(bbb_idx)] = "BBB"

gene_to_index["SAScore"] = int(sascore_idx)
gene_to_index["CNSMPO"] = int(cnsmpo_idx)
gene_to_index["BBB"] = int(bbb_idx)

with open("../data/target_conditions_to_index_mpo.json", "w") as f:
    json.dump({
        "idx_to_gene": idx_to_gene, 
        "gene_to_index": gene_to_index
    }, f, indent=4, ensure_ascii=False)